# Multi-resolution Event V2 — Relation V2 on one P100

Select **P100**, keep Internet off, and choose **Save & Run All**. Keep the Notebook slug exactly `vanila111/trauma-predict-relation-v2-p100-r9`. The first successful run stops after the complete step-250 validation/checkpoint proof; later Save Runs restore the prior hash-bound output and advance through 1500, 2750, 4000, then resumable free-running evaluation. The Notebook uses only the frozen private bundle, never clones source, rebuilds data, resumes v8, or runs an ablation.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

DATASET_REF = 'vanila111/trauma-predict-relation-v2-p100-r9-bundle'
SCHEMA = 'trauma_predict.multires_event_v2_relation_v2_p100_bundle.v1'

def find_bundles(root):
    matches = []
    if not root.is_dir():
        return matches
    for manifest_path in sorted(root.rglob('run_bundle_manifest.json')):
        if manifest_path.is_symlink() or not manifest_path.is_file():
            continue
        payload = json.loads(manifest_path.read_text(encoding='utf-8'))
        if payload.get('schema') == SCHEMA and payload.get('dataset_ref') == DATASET_REF:
            matches.append(manifest_path.parent.resolve())
    return matches

def require_one_bundle(root):
    matches = find_bundles(root)
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one Relation V2 P100 bundle under {root}, found {len(matches)}')
    return matches[0]

import torch
if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
    raise RuntimeError(f'Select one Kaggle P100; found {torch.cuda.device_count()} visible GPU(s)')
device_name = torch.cuda.get_device_name(0)
if 'P100' not in device_name.upper():
    raise RuntimeError(f'Select Kaggle P100; current device is {device_name!r}')
print('RELATION_V2_P100_GPU_PREFLIGHT_OK', device_name, flush=True)

attached = find_bundles(Path('/kaggle/input'))
if len(attached) > 1:
    raise RuntimeError(f'Expected at most one attached Relation V2 bundle, found {attached}')
if attached:
    bundle = attached[0]
    print('RELATION_V2_P100_AUTOMATIC_INPUT_OK', bundle, flush=True)
else:
    try:
        import kagglehub
    except ImportError as exc:
        raise RuntimeError('Kaggle image lacks kagglehub; private bundle cannot be resolved') from exc
    try:
        downloaded = kagglehub.dataset_download(DATASET_REF, force_download=True)
    except Exception as exc:
        raise RuntimeError(f'Authenticated private Dataset download failed: {DATASET_REF}') from exc
    if not downloaded:
        raise RuntimeError('kagglehub returned no private Dataset path')
    bundle = require_one_bundle(Path(downloaded).resolve())
    print('RELATION_V2_P100_PRIVATE_DATASET_DOWNLOAD_OK', bundle, flush=True)

launcher = bundle / 'run_relation_v2_p100_bundle.py'
if launcher.is_symlink() or not launcher.is_file():
    raise FileNotFoundError(f'Bundle lacks its manifest-bound launcher: {launcher}')
subprocess.run([sys.executable, str(launcher), '--bundle-root', str(bundle)], check=True)
